# Round 4: Sparse Autoencoder — what did fine-tuning change *inside* the model?

**The story so far:** fine-tuning took Qwen2.5-0.5B (base) from 6.5% → 33% on GSM8K. Behaviorally, we know it learned. Mechanistically, we know nothing. This round cracks the box open.

**The method (a scaled-down version of Anthropic's interpretability approach):**
1. Run the same math problems through the **base model** and the **fine-tuned model**
2. Capture internal activations from a middle layer (the residual stream)
3. Train **one shared sparse autoencoder** on both sets of activations — a small 2-layer network that re-expresses the 896-dim activations as ~7,000 sparse, hopefully-interpretable features
4. For each feature, compare how strongly it fires in base vs fine-tuned — features with the biggest shift are *what fine-tuning changed*
5. Look at the top-activating tokens of shifted features to interpret them

**Honest expectations:** at 0.5B scale with a small SAE trained on modest data, some features will be interpretable, many will be mush. Finding even a handful of clearly math-flavored features that fine-tuning amplified would be a success.

Setup: T4 GPU, Run all. ~40–60 min total (including retraining the fine-tuned adapter if it's not present from round 3).

In [1]:
# Cell 1: Install
!pip uninstall -y -q torchao
!pip install -q -U transformers datasets peft accelerate

import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00
GPU: Tesla T4


In [2]:
# Cell 2: Data + tokenizer (same setup as round 3)
import random, re
from datasets import load_dataset
from transformers import AutoTokenizer

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
MAX_LEN = 640

torch.manual_seed(SEED)
gsm = load_dataset("openai/gsm8k", "main")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_prompt(question):
    return f"Question: {question}\nAnswer:"

train_pool = list(gsm["train"])
random.Random(SEED).shuffle(train_pool)
train_pool = train_pool[:2000]                 # round-3 training data
probe_pool = list(gsm["test"])[:300]           # activation-capture set (held-out)
print(f"Probe set: {len(probe_pool)} problems")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Probe set: 300 problems


In [3]:
# Cell 3: Get the fine-tuned adapter — load if present, otherwise retrain run A (~20 min)
import os, gc
from torch.utils.data import Dataset, SequentialSampler
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

ADAPTER = "./adapter_r3_A_shuffled"

if not os.path.exists(ADAPTER):
    print("Adapter not found — retraining round-3 run A (shuffled)...")

    def tokenize_masked(ex):
        answer = re.sub(r"<<[^>]*>>", "", ex["answer"])
        prompt = make_prompt(ex["question"])
        full = prompt + " " + answer + tokenizer.eos_token
        prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_LEN)["input_ids"]
        enc = tokenizer(full, truncation=True, max_length=MAX_LEN)
        labels = list(enc["input_ids"])
        n = min(len(prompt_ids), len(labels))
        labels[:n] = [-100] * n
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

    order_A = list(train_pool)
    random.Random(SEED + 1).shuffle(order_A)
    dataset_A = [tokenize_masked(e) for e in order_A]

    class ListDataset(Dataset):
        def __init__(self, items): self.items = items
        def __len__(self): return len(self.items)
        def __getitem__(self, i): return self.items[i]

    def collate_masked(batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id
        ii, am, lb = [], [], []
        for x in batch:
            n = max_len - len(x["input_ids"])
            ii.append(list(x["input_ids"]) + [pad_id] * n)
            am.append(list(x["attention_mask"]) + [0] * n)
            lb.append(list(x["labels"]) + [-100] * n)
        return {"input_ids": torch.tensor(ii), "attention_mask": torch.tensor(am), "labels": torch.tensor(lb)}

    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model.config.use_cache = False
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
    model = get_peft_model(model, lora)
    args = TrainingArguments(output_dir="./out_r4", num_train_epochs=1,
                             per_device_train_batch_size=4, gradient_accumulation_steps=2,
                             learning_rate=1e-4, lr_scheduler_type="constant", warmup_steps=10,
                             logging_steps=100, save_strategy="no", fp16=True, report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(dataset_A),
                      data_collator=collate_masked)  # order doesn't matter now — round 3 proved it
    trainer.train()
    model.save_pretrained(ADAPTER)
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()

print("✓ Fine-tuned adapter ready at", ADAPTER)

Adapter not found — retraining round-3 run A (shuffled)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Step,Training Loss
100,0.575379
200,0.553547


✓ Fine-tuned adapter ready at ./adapter_r3_A_shuffled


In [4]:
# Cell 4: Capture activations from the middle of both models on the SAME texts
# We hook the residual stream after layer 12 (of 24). Each token position gives one 896-dim vector.
from peft import PeftModel

LAYER = 12
MAX_PROBE_TOKENS = 200   # per problem

def capture_activations(use_adapter):
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model = PeftModel.from_pretrained(base, ADAPTER) if use_adapter else base
    model.eval()

    captured = []
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        captured.append(h.detach().float().cpu())

    # find the decoder layer list on either the plain or the PEFT-wrapped model
    inner = model
    while not hasattr(inner, "layers"):
        for attr in ("base_model", "model"):
            if hasattr(inner, attr):
                inner = getattr(inner, attr)
                break
        else:
            raise RuntimeError("couldn't find decoder layers")
    handle = inner.layers[LAYER].register_forward_hook(hook)

    all_vecs, all_tokens = [], []   # activation vectors + (problem_idx, token_string) tags
    with torch.no_grad():
        for pi, ex in enumerate(probe_pool):
            answer = re.sub(r"<<[^>]*>>", "", ex["answer"])
            text = make_prompt(ex["question"]) + " " + answer
            enc = tokenizer(text, return_tensors="pt", truncation=True,
                            max_length=MAX_PROBE_TOKENS).to("cuda")
            captured.clear()
            model(**enc)
            acts = captured[0][0]                     # [seq_len, 896]
            toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
            all_vecs.append(acts)
            all_tokens.extend((pi, t) for t in toks)
            if (pi + 1) % 100 == 0:
                print(f"  {pi+1}/{len(probe_pool)} problems")

    handle.remove()
    del model, base
    gc.collect(); torch.cuda.empty_cache()
    return torch.cat(all_vecs, dim=0), all_tokens

print("Capturing BASE model activations...")
acts_base, tokens_base = capture_activations(use_adapter=False)
print("Capturing FINE-TUNED model activations...")
acts_ft, tokens_ft = capture_activations(use_adapter=True)

assert len(tokens_base) == len(tokens_ft), "token streams should be identical"
print(f"\nCaptured {acts_base.shape[0]:,} token activations per model, dim {acts_base.shape[1]}")

Capturing BASE model activations...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  100/300 problems
  200/300 problems
  300/300 problems
Capturing FINE-TUNED model activations...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  100/300 problems
  200/300 problems
  300/300 problems

Captured 44,125 token activations per model, dim 896


In [5]:
# Cell 5: Quick sanity metric before the SAE — how much did activations move overall?
import torch.nn.functional as F
cos = F.cosine_similarity(acts_base, acts_ft, dim=1)
print(f"Per-token cosine similarity base vs fine-tuned:")
print(f"  mean {cos.mean():.4f} | 10th pct {cos.quantile(0.10):.4f} | min {cos.min():.4f}")
print("\n(1.0 = identical. LoRA is a small nudge, so expect high similarity —")
print(" the interesting question is WHERE the small differences live, which is the SAE's job.)")

Per-token cosine similarity base vs fine-tuned:
  mean 0.9925 | 10th pct 0.9863 | min 0.8563

(1.0 = identical. LoRA is a small nudge, so expect high similarity —
 the interesting question is WHERE the small differences live, which is the SAE's job.)


In [6]:
# Cell 6: Train ONE shared SAE on both models' activations (~5-10 min)
# Shared dictionary => features are directly comparable between the two models.
import torch.nn as nn

D_IN = acts_base.shape[1]        # 896
D_SAE = D_IN * 8                 # 7168 features
L1_COEF = 5e-4
EPOCHS = 15
BATCH = 4096

class SAE(nn.Module):
    def __init__(self, d_in, d_sae):
        super().__init__()
        self.enc = nn.Linear(d_in, d_sae)
        self.dec = nn.Linear(d_sae, d_in)
    def forward(self, x):
        f = torch.relu(self.enc(x))
        return self.dec(f), f

# normalize activations so L1 coefficient behaves predictably
all_acts = torch.cat([acts_base, acts_ft], dim=0)
scale = all_acts.norm(dim=1).mean()
train_data = (all_acts / scale).cuda()

torch.manual_seed(SEED)
sae = SAE(D_IN, D_SAE).cuda()
opt = torch.optim.Adam(sae.parameters(), lr=1e-3)

n = train_data.shape[0]
for epoch in range(EPOCHS):
    perm = torch.randperm(n, device="cuda")
    tot_mse, tot_l1, steps = 0.0, 0.0, 0
    for i in range(0, n, BATCH):
        x = train_data[perm[i:i+BATCH]]
        recon, feats = sae(x)
        mse = F.mse_loss(recon, x)
        l1 = feats.abs().mean()
        loss = mse + L1_COEF * l1
        opt.zero_grad(); loss.backward(); opt.step()
        tot_mse += mse.item(); tot_l1 += l1.item(); steps += 1
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch+1}/{EPOCHS}  mse {tot_mse/steps:.5f}  l1 {tot_l1/steps:.4f}")

# compute feature activations for each model separately
with torch.no_grad():
    _, feats_base = sae((acts_base / scale).cuda()); feats_base = feats_base.cpu()
    _, feats_ft   = sae((acts_ft / scale).cuda());   feats_ft = feats_ft.cpu()

alive = ((feats_base.max(0).values > 1e-4) | (feats_ft.max(0).values > 1e-4)).sum().item()
sparsity = (feats_ft > 1e-4).float().mean().item()
print(f"\nAlive features: {alive}/{D_SAE}  |  avg fraction active per token: {sparsity:.3%}")
print("(healthy SAE: most features dead most of the time, a few percent active per token)")

epoch 5/15  mse 0.00009  l1 0.0053
epoch 10/15  mse 0.00004  l1 0.0060
epoch 15/15  mse 0.00003  l1 0.0062

Alive features: 6566/7168  |  avg fraction active per token: 19.415%
(healthy SAE: most features dead most of the time, a few percent active per token)


In [7]:
# Cell 7: THE ANALYSIS — which features did fine-tuning amplify or suppress?
mean_base = feats_base.mean(0)
mean_ft = feats_ft.mean(0)
shift = mean_ft - mean_base                      # + amplified by FT, - suppressed

def show_feature(fi, k=12):
    """Show the top-activating tokens (with context) for feature fi in the FT model."""
    vals = feats_ft[:, fi]
    top = vals.topk(k).indices.tolist()
    seen, lines = set(), []
    for t in top:
        pi, tok = tokens_ft[t]
        ctx = "".join(x[1] for x in tokens_ft[max(0, t-6):t+3] if x[0] == pi)
        ctx = ctx.replace("\u0120", " ").replace("\u010a", "\\n")
        key = (tok, ctx[:30])
        if key in seen: continue
        seen.add(key)
        lines.append(f"      act {vals[t]:.2f} | …{ctx}…")
    return lines[:6]

print("="*70)
print("TOP 15 FEATURES AMPLIFIED BY FINE-TUNING (the 'learned math' candidates)")
print("="*70)
for rank, fi in enumerate(shift.topk(15).indices.tolist(), 1):
    print(f"\n#{rank}  feature {fi}  (mean act: base {mean_base[fi]:.3f} → ft {mean_ft[fi]:.3f})")
    for line in show_feature(fi):
        print(line)

print("\n" + "="*70)
print("TOP 10 FEATURES SUPPRESSED BY FINE-TUNING")
print("="*70)
for rank, fi in enumerate((-shift).topk(10).indices.tolist(), 1):
    print(f"\n#{rank}  feature {fi}  (mean act: base {mean_base[fi]:.3f} → ft {mean_ft[fi]:.3f})")
    for line in show_feature(fi):
        print(line)

TOP 15 FEATURES AMPLIFIED BY FINE-TUNING (the 'learned math' candidates)

#1  feature 5291  (mean act: base 0.038 → ft 0.041)
      act 0.12 | … pieces of pie had been taken by guests.\n…
      act 0.12 | … is a freelance blogger who writes about health topics…
      act 0.12 | …000 a year for miles driven\n…
      act 0.12 | … takes 10 minutes to cover every …
      act 0.12 | …5 eggs a day. Every day Jerry collects…
      act 0.12 | … a total of 30*=90…

#2  feature 5386  (mean act: base 0.039 → ft 0.042)
      act 0.16 | …: Dorothy has 12 / 3…
      act 0.15 | … edits 1000 / 2…
      act 0.15 | … be 1245 / 1…
      act 0.15 | … He bought 200 / 4…
      act 0.15 | … younger son washes 8 / 2…
      act 0.15 | … collar takes 900 / 1…

#3  feature 1331  (mean act: base 0.049 → ft 0.052)
      act 0.19 | … 2100 spots total.\nTogether…
      act 0.19 | … the next two days.\nThe total number of…
      act 0.19 | …1000 grams in total.\nThe…
      act 0.18 | …10 windows\nSo the total number 

In [8]:
# Cell 8: How to read the output above
print("""
HOW TO INTERPRET WHAT YOU'RE SEEING
-----------------------------------
Each 'feature' is one direction the SAE found in the model's internal
activations. For each, you see the text contexts where it fires hardest.

Ask of each amplified feature: do its top contexts share a THEME?
  - numbers / digits            -> arithmetic representation strengthened
  - '=', '+', 'total', 'each'   -> calculation/aggregation machinery
  - '####', 'Answer'            -> answer-format feature (learned our format!)
  - names / objects from word problems -> entity tracking
  - no visible theme            -> uninterpretable at this scale (expected for many)

The '####' one is worth hunting for — if a feature exists that fires on the
answer delimiter mainly in the fine-tuned model, you've literally watched
fine-tuning install a new behavior as a visible internal feature.

Honest caveats: a small SAE on 100k-ish tokens is the toy end of this
technique. Feature quality will be mixed. Anthropic's versions use billions
of activations and millions of features. But the method is the same one.

Paste interesting (or confusing) features back to Claude for a second opinion.
""")


HOW TO INTERPRET WHAT YOU'RE SEEING
-----------------------------------
Each 'feature' is one direction the SAE found in the model's internal
activations. For each, you see the text contexts where it fires hardest.

Ask of each amplified feature: do its top contexts share a THEME?
  - numbers / digits            -> arithmetic representation strengthened
  - '=', '+', 'total', 'each'   -> calculation/aggregation machinery
  - '####', 'Answer'            -> answer-format feature (learned our format!)
  - names / objects from word problems -> entity tracking
  - no visible theme            -> uninterpretable at this scale (expected for many)

The '####' one is worth hunting for — if a feature exists that fires on the
answer delimiter mainly in the fine-tuned model, you've literally watched
fine-tuning install a new behavior as a visible internal feature.

Honest caveats: a small SAE on 100k-ish tokens is the toy end of this
technique. Feature quality will be mixed. Anthropic's versions u

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
